In [5]:
import sys 
from pathlib import Path
# repo root 
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd 
import numpy as np 


%load_ext autoreload
%autoreload 2
RAW_PATH = Path("../data/raw/kddcup.csv")
NAMES_PATH = Path("../data/raw/kddcup.names.txt")
Path("../src/__init__.py").touch()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#peek or read the csv

from IPython.display import display
df_peek = pd.read_csv(RAW_PATH, header=None, nrows=3)
display(df_peek)


,0,1,2,3,4,5,6,7,8,9,...,32,33,34,35,36,37,38,39,40,41
0,0,tcp,http,SF,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.


In [ ]:
# 2. Load raw dataset via src.data and validate schema via src.validate
from src.data import load_raw
from src.validate import validate_raw

df = load_raw()
validate_raw(df)

# Test assertion
assert df.shape == (494021, 42), f'Expected (494021, 42), got {df.shape}'
print(f'Shape: {df.shape}')


In [ ]:
object_cols = df.select_dtypes(include=['object']).columns.tolist()
distinct_labels = df['label'].nunique()
ends_with_dot = df['label'].str.endswith('.').any()


print(f'Object columns: {object_cols}')
print(f'Distinct labels: {distinct_labels}')
print(f'Any end with ".": {ends_with_dot}')


In [16]:
attack_counts = df['label'].value_counts()
display(attack_counts.head(10))

label
smurf.          280790
neptune.        107201
normal.          97278
back.             2203
satan.            1589
ipsweep.          1247
portsweep.        1040
warezclient.      1020
teardrop.          979
pod.               264
Name: count, dtype: int64

In [17]:
num_duplicates = df.duplicated().sum()
total_rows = len(df)
dup_share = num_duplicates / total_rows

print(f"Exact duplicates: {num_duplicates:,}")
print(f"Share of file: {dup_share:.2%}")

# Duplicate rate per attack (duplicate copies / total for that label)
dup_per_attack = (df.duplicated().groupby(df["label"]).sum() / df["label"].value_counts()).sort_values(ascending=False)
display(dup_per_attack.head(5))


Exact duplicates: 348,435
Share of file: 70.53%


label
smurf.      280335
neptune.     71652
normal.      11682
back.         1365
satan.         877
dtype: int64

In [18]:
has_nan = df.isna().sum().sum()
numeric_df = df.select_dtypes(include=[np.number])
has_inf = np.isinf(numeric_df).sum().sum()

print(f"NaN count: {has_nan}")
print(f"Inf count: {has_inf}")

NaN count: 0
Inf count: 0


In [19]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"Constant columns: {constant_cols}")

Constant columns: ['num_outbound_cmds', 'is_host_login']


In [21]:
rate_cols = [col for col in df.columns if col.endswith('_rate')]
rate_min = df[rate_cols].min().min()
rate_max = df[rate_cols].max().max()


numeric_df = df.select_dtypes(include=[np.number])
has_negative = (numeric_df < 0).sum().sum()

su_vals = sorted(df['su_attempted'].unique())

print(f"Rate columns range: [{rate_min}, {rate_max}]")
print(f"Negative values count: {has_negative}")
print(f"`su_attempted` unique values: {su_vals}")

Rate columns range: [0.0, 1.0]
Negative values count: 0
`su_attempted` unique values: [np.int64(0), np.int64(1), np.int64(2)]


In [22]:
protocol_cats = df['protocol_type'].nunique()
service_cats = df['service'].nunique()
flag_cats = df['flag'].nunique()

print(f"protocol_type categories: {protocol_cats}")
print(f"service categories: {service_cats}")
print(f"flag categories: {flag_cats}")

protocol_type categories: 3
service categories: 66
flag categories: 11


In [ ]:
# 10. If a model always predicted the most common label, what accuracy?
most_common_label = df['label'].value_counts().idxmax()
most_common_count = df['label'].value_counts().max()
accuracy = most_common_count / len(df)

print(f"Most common label: {most_common_label}")
print(f"Accuracy if always predicting '{most_common_label}': {accuracy:.2%}")

In [24]:
attack_types_path = Path("../data/raw/training_attack_types.txt")
assert attack_types_path.exists(), (
    f"{attack_types_path} not found. Download it from the KDD Cup 1999 data "
    f"page into data/raw/ before continuing."
)

FAMILY_MAP = {
    "dos": "DoS",
    "probe": "Probe",
    "r2l": "R2L",
    "u2r": "U2R",
}

attack_to_family = {}
with open(attack_types_path) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            attack, family = parts
            attack_to_family[attack] = FAMILY_MAP.get(family.lower(), family)

attack_to_family["normal"] = "Normal"   # 'normal' is not listed in the attacks file

print(f"Mappings built from file: {len(attack_to_family)}")
print(f"Families present: {sorted(set(attack_to_family.values()))}")
assert sorted(set(attack_to_family.values())) == ["DoS", "Normal", "Probe", "R2L", "U2R"], \
    "Unexpected family names in training_attack_types.txt"
print("Built mapping from file ")

from src.config import ATTACK_TO_FAMILY
assert attack_to_family == ATTACK_TO_FAMILY, "attack_to_family does not match config.ATTACK_TO_FAMILY"
print("Mapping in config matches training_attack_types.txt exactly!")


Mappings built from file: 23
Families present: ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']
Built mapping from file 


In [ ]:
# Phase 2: Test the validator with deliberately broken inputs
# 1. Inject NaN
df_nan = df.copy()
df_nan.iloc[0, 0] = np.nan
try:
    validate_raw(df_nan)
except AssertionError as e:
    print(f"[PASS] Caught NaN: {e}")
else:
    raise RuntimeError("Validator did NOT catch NaN!")

# 2. Rename a column
df_rename = df.copy()
df_rename = df_rename.rename(columns={df.columns[0]: "wrong_column_name"})
try:
    validate_raw(df_rename)
except AssertionError as e:
    print(f"[PASS] Caught column mismatch: {str(e)[:50]}...")
else:
    raise RuntimeError("Validator did NOT catch column mismatch!")

# 3. Add an unknown label
df_unknown = df.copy()
df_unknown.iloc[0, df.columns.get_loc("label")] = "unknown_attack_type"
try:
    validate_raw(df_unknown)
except AssertionError as e:
    print(f"[PASS] Caught unknown label: {e}")
else:
    raise RuntimeError("Validator did NOT catch unknown label!")

# 4. Add a trailing dot on a known attack label to isolate the dot check
df_dot = df.copy()
df_dot.iloc[0, df.columns.get_loc("label")] = "normal."
try:
    validate_raw(df_dot)
except AssertionError as e:
    assert "ending with dot" in str(e), f"Expected dot check to fire, got: {e}"
    print(f"[PASS] Caught trailing dot: {e}")
else:
    raise RuntimeError("Validator did NOT catch trailing dot!")

print("All 4 broken input tests caught as expected!")


Notebook 02 — "Part C: Preprocessing tests"

In [ ]:
import sys 
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))


import numpy as np
import pandas as pd 
%load_ext autoreload
%autoreload 2

from src.data import load_train
from src.config import CATEGORICAL_COLS, BINARY_COLS, NUMERIC_COLS, HEAVY_TAILED_COLS
from src.features import add_features, ENGINEERED_COLS, CONTENT_COLS
from src.preprocessing import build_preprocessor, build_pipeline

X_train, y_train, meta_train = load_train()
print(f"X_train: {X_train.shape}")
print(f"CATEGORICAL={len(CATEGORICAL_COLS)}, BINARY={len(BINARY_COLS)}, "
      f"NUMERIC={len(NUMERIC_COLS)} (heavy-tailed={len(HEAVY_TAILED_COLS)})")

In [ ]:


X_train_eng = add_features(X_train)

configs = [
    (False, False, X_train),
    (False, True,  X_train_eng),
    (True,  False, X_train),
    (True,  True,  X_train_eng),
]


for scale, eng, Xin in configs:
    pre = build_preprocessor(scale=scale, engineered=eng).fit(Xin)
    out = pre.transform(Xin)
    print(f"scale={scale!s:5} engineered={eng!s:5} -> {out.shape[1]:3d} output columns")

In [ ]:
for scale, eng, Xin in configs:
    pre = build_preprocessor(scale=scale, engineered=eng).fit(Xin)
    out = pre.transform(Xin)
    assert not np.isnan(out).any(), f"NaN found (scale={scale}, eng={eng})"
    assert not np.isinf(out).any(), f"inf found (scale={scale}, eng={eng})"
    assert out.shape[1] == len(pre.get_feature_names_out()), \
        f"width mismatch (scale={scale}, eng={eng})"
print(" Test 1: no NaN/inf, width == len(get_feature_names_out())")

In [ ]:
def get_assigned_cols(pre):
    assigned = set()
    for name, trans, cols in pre.transformers:
        if cols is None:
            continue
        assigned.update(cols if isinstance(cols, (list, tuple)) else [cols])
    return assigned

# Base groups must be disjoint
base = list(CATEGORICAL_COLS) + list(BINARY_COLS) + list(NUMERIC_COLS)
assert len(base) == len(set(base)), "base column groups overlap!"

# engineered=False: assigned == all raw columns
pre_ne = build_preprocessor(scale=False, engineered=False)
assert get_assigned_cols(pre_ne) == set(X_train.columns), "coverage mismatch (engineered=False)"

# engineered=True: assigned == raw columns + engineered columns
pre_e = build_preprocessor(scale=True, engineered=True)
assert get_assigned_cols(pre_e) == set(X_train.columns) | set(ENGINEERED_COLS), \
    "coverage mismatch (engineered=True)"
print(" Test 2: every input column lands in exactly one transformer group")

In [ ]:
X_unseen = X_train.head().copy()
X_unseen.loc[X_unseen.index[0], "service"] = "never_seen_service"
pre = build_preprocessor(scale=False, engineered=False).fit(X_train)
out = pre.transform(X_unseen)   # handle_unknown='ignore' -> no error
assert out.shape[0] == 5
print("Test 3: unseen category handled without error")

In [ ]:
pre_scaled = build_preprocessor(scale=True, engineered=False).fit(X_train)
X_tr_scaled = pre_scaled.transform(X_train)
fnames = list(pre_scaled.get_feature_names_out())

scaled_idx = [i for i, n in enumerate(fnames) if n.startswith(("heavy__", "num__"))]
scaled_block = X_tr_scaled[:, scaled_idx]
assert np.allclose(scaled_block.mean(axis=0), 0, atol=1e-8), "scaled mean != 0"
assert np.allclose(scaled_block.std(axis=0), 1, atol=1e-8), "scaled std != 1"

# The fitted mean must equal the TRAIN mean (proves no test leakage)
other_numeric = [c for c in NUMERIC_COLS if c not in HEAVY_TAILED_COLS]
num_scaler = pre_scaled.named_transformers_["num"]
assert np.allclose(num_scaler.mean_, X_train[other_numeric].mean().values)

heavy_scaler = pre_scaled.named_transformers_["heavy"].named_steps["scaler"]
assert np.allclose(heavy_scaler.mean_, np.log1p(X_train[HEAVY_TAILED_COLS]).mean().values)
print("Test 4: scaled cols mean≈0/std≈1; scaler fitted on train mean")

In [ ]:
pre_ns = build_preprocessor(scale=False, engineered=False).fit(X_train)
X_tr_ns = pre_ns.transform(X_train)
fnames = list(pre_ns.get_feature_names_out())

other_numeric = [c for c in NUMERIC_COLS if c not in HEAVY_TAILED_COLS]
for prefix, cols in [("binary__", BINARY_COLS),
                     ("heavy__", HEAVY_TAILED_COLS),
                     ("num__", other_numeric)]:
    idx = [i for i, n in enumerate(fnames) if n.startswith(prefix)]
    assert np.array_equal(X_tr_ns[:, idx], X_train[cols].values), f"passthrough mismatch: {prefix}"
print("Test 5: scale=False passthrough columns equal raw values")

In [ ]:
X_small = X_train.head(3).copy()
X_eng = add_features(X_small)

for i in range(3):
    row = X_small.iloc[i]
    assert np.isclose(X_eng["bytes_ratio"].iloc[i],
                      np.log1p(row["src_bytes"]) - np.log1p(row["dst_bytes"]))
    assert X_eng["zero_payload"].iloc[i] == int(row["src_bytes"] == 0 and row["dst_bytes"] == 0)
    assert X_eng["content_activity"].iloc[i] == sum(row[c] for c in CONTENT_COLS)
    assert X_eng["privilege_flag"].iloc[i] == int(
        row["root_shell"] > 0 or row["su_attempted"] > 0 or row["num_root"] > 0)
print("Test 6: engineered features match hand computation")

In [ ]:
X_before = X_train.head(3).copy()
X_snapshot = X_before.copy(deep=True)
_ = add_features(X_before)
assert X_before.equals(X_snapshot), "add_features mutated its input!"
assert set(X_before.columns) == set(X_snapshot.columns)
print("Test 7: add_features does not mutate its input")

In [ ]:
X_batch = X_train.head(5)
X_eng_batch = add_features(X_batch)
for i in range(5):
    X_single = add_features(X_batch.iloc[[i]])
    for col in ENGINEERED_COLS:
        assert np.isclose(X_eng_batch[col].iloc[i], X_single[col].iloc[0]), \
            f"row-independence violated for {col}"
print("Test 8: single-row transform matches batch transform (row-wise, no leakage)")

In [ ]:
from sklearn.dummy import DummyClassifier
for scale in [False, True]:
    for eng in [False, True]:
        pipe = build_pipeline(DummyClassifier(strategy="most_frequent"),
                              scale=scale, engineered=eng)
        pipe.fit(X_train, y_train)
        _ = pipe.predict(X_train.head())
        print(f"pipeline scale={scale!s:5} engineered={eng!s:5} OK  "
              f"steps={[n for n, _ in pipe.steps]}")
print("all pipeline configs fit and predict")